# FIT5196 Assessment 1 - Solution Notebook — Task 1 Version

**Group:** Group050


**Members:** <br>
Yu Wang ID:  ; <br>
Xingao Zhan ID: 36354775 <br>
Keshu Zhang ID:   ; <br>
Qingzhuo Zhao ID: 26662841

**Stage label:** Task 1 — structured parsing, source profiling and source-to-target mapping  
**Version date:** 2026-08-22  
**Status:** Task 1 complete — source-profile checks PASS; mapping audit 12/12 PASS  
**Scope:** This stage version completes Task 1 only. Tasks 2–6 remain outside this checkpoint.


## 0. Configuration and reproducibility

Keep all configurable paths in this section. The final notebook must run with
**Restart and Run All** without manual file edits or network access.


In [2]:
from pathlib import Path
import sys

GROUP_ID = "Group050"
PROJECT_DIR = Path.cwd()

# Official default: support files and raw_input are beside the notebook.
# Development fallback: keep the supplied package read-only in the sibling
# Group050_A1 folder while all generated and edited files stay here.
PACKAGE_DIR = PROJECT_DIR
LOCAL_PACKAGE_DIR = PROJECT_DIR.parent / "Group050_A1"
if not (PACKAGE_DIR / "raw_input").is_dir() and (LOCAL_PACKAGE_DIR / "raw_input").is_dir():
    PACKAGE_DIR = LOCAL_PACKAGE_DIR

INPUT_DIR = PACKAGE_DIR / "raw_input"
OUTPUT_DIR = PROJECT_DIR / "outputs"
TEMPLATE_DIR = PACKAGE_DIR / "templates"

JSON_PATH = INPUT_DIR / f"{GROUP_ID}_commerce.json"
XML_PATH = INPUT_DIR / f"{GROUP_ID}_operations.xml"
DATA_DICTIONARY_PATH = PACKAGE_DIR / "public_data_dictionary.csv"
MAPPING_TEMPLATE_PATH = TEMPLATE_DIR / "A1_source_to_target_mapping_template.csv"
MAPPING_OUTPUT_PATH = PROJECT_DIR / "Task 01 Data" / f"{GROUP_ID}_source_to_target_mapping.csv"
PUBLIC_TEXT_TESTS_PATH = TEMPLATE_DIR / "A1_public_text_test_cases.csv"

required_paths = {
    "JSON source": JSON_PATH,
    "XML source": XML_PATH,
    "public data dictionary": DATA_DICTIONARY_PATH,
    "mapping template": MAPPING_TEMPLATE_PATH,
    "public text tests": PUBLIC_TEXT_TESTS_PATH,
}
missing_paths = {
    label: path for label, path in required_paths.items() if not path.is_file()
}
if missing_paths:
    missing_text = "\n".join(
        f"- {label}: {path}" for label, path in missing_paths.items()
    )
    raise FileNotFoundError(f"Required assignment files are missing:\n{missing_text}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

package_display = "." if PACKAGE_DIR == PROJECT_DIR else "../Group050_A1"
print("Working directory: .")
print(f"Read-only package directory: {package_display}")
print(f"JSON source: {JSON_PATH.name}")
print(f"XML source: {XML_PATH.name}")
print("Output directory: outputs")

Working directory: .
Read-only package directory: .
JSON source: Group050_commerce.json
XML source: Group050_operations.xml
Output directory: outputs


### 0.1 Environment and dependencies

Import the libraries used by your submitted workflow. Record non-standard
dependencies in `requirements.txt`.


In [3]:
# EVIDENCE: SEC-0.1-ENVIRONMENT
import json
import platform
import sys
import xml.etree.ElementTree as ET
from collections import Counter

import numpy as np
import pandas as pd

def show(frame, title=None):
    if title:
        print(f"\n{title}")
    print(frame.to_string(index=False))

environment = pd.DataFrame(
    [
        {"component": "Python", "version": platform.python_version()},
        {"component": "pandas", "version": pd.__version__},
        {"component": "NumPy", "version": np.__version__},
        {"component": "XML parser", "version": "xml.etree.ElementTree (stdlib)"},
        {"component": "JSON parser", "version": "json (stdlib)"},
    ]
)
show(environment, "Environment and dependencies")



Environment and dependencies
  component                        version
     Python                        3.12.13
     pandas                          2.2.3
      NumPy                          2.1.3
 XML parser xml.etree.ElementTree (stdlib)
JSON parser                  json (stdlib)


## 1. Parse and profile the two sources

Use structured JSON and XML parsers. Record source grains, nested/repeated
structures, candidate keys, formats, missing-value conventions and evidence of
within-source or cross-source overlap.


### 1.1 JSON structure and profile


In [4]:
# EVIDENCE: SEC-1.1-JSON-PROFILE
with JSON_PATH.open(encoding="utf-8") as handle:
    json_source = json.load(handle)

json_customers = json_source["customerProfiles"]
json_orders = json_source["orders"]
json_items = [item for order in json_orders for item in order["shoppingCart"]]
json_deliveries = [order["delivery"] for order in json_orders]
json_reviews = json_source["productReviews"]

def profile_records(source_collection, grain, records, key_field):
    keys = [record.get(key_field) for record in records]
    non_missing = [key for key in keys if key not in (None, "")]
    frequencies = Counter(non_missing)
    fingerprints = Counter(
        json.dumps(record, sort_keys=True, ensure_ascii=False) for record in records
    )
    return {
        "source_collection": source_collection,
        "grain": grain,
        "candidate_key": key_field,
        "rows": len(records),
        "missing_key": len(keys) - len(non_missing),
        "unique_key": len(frequencies),
        "duplicate_key_groups": sum(count > 1 for count in frequencies.values()),
        "duplicate_extra_rows": len(non_missing) - len(frequencies),
        "exact_duplicate_groups": sum(count > 1 for count in fingerprints.values()),
        "exact_duplicate_extra_rows": len(records) - len(fingerprints),
    }

json_profile = pd.DataFrame(
    [
        profile_records("customerProfiles[]", "one customer profile", json_customers, "customerID"),
        profile_records("orders[]", "one source order", [order["header"] for order in json_orders], "orderID"),
        profile_records("orders[].shoppingCart[]", "one source order item", json_items, "orderItemID"),
        profile_records("orders[].delivery", "one source delivery", json_deliveries, "deliveryID"),
        profile_records("productReviews[]", "one source review", json_reviews, "reviewID"),
    ]
)
show(json_profile, "JSON collection profile")

json_structure = pd.DataFrame(
    [
        {"path": "customerProfiles[]", "representation": "repeated root array", "fields": ", ".join(json_customers[0].keys())},
        {"path": "orders[]", "representation": "repeated root array with header, shoppingCart[] and delivery", "fields": ", ".join(json_orders[0].keys())},
        {"path": "orders[].header", "representation": "nested object", "fields": ", ".join(json_orders[0]["header"].keys())},
        {"path": "orders[].shoppingCart[]", "representation": "nested repeated array", "fields": ", ".join(json_items[0].keys())},
        {"path": "orders[].delivery", "representation": "nested object", "fields": ", ".join(json_deliveries[0].keys())},
        {"path": "productReviews[]", "representation": "repeated root array", "fields": ", ".join(json_reviews[0].keys())},
    ]
)
show(json_structure, "JSON nesting and fields")

first_header = json_orders[0]["header"]
first_delivery = json_orders[0]["delivery"]
json_formats = pd.DataFrame(
    [
        {"concept": "timestamp", "path": "orders[].header.orderTimestamp", "example": repr(first_header["orderTimestamp"]), "Python type": type(first_header["orderTimestamp"]).__name__},
        {"concept": "date", "path": "orders[].delivery.dispatchDate", "example": repr(first_delivery["dispatchDate"]), "Python type": type(first_delivery["dispatchDate"]).__name__},
        {"concept": "boolean", "path": "orders[].header.expeditedDelivery", "example": repr(first_header["expeditedDelivery"]), "Python type": type(first_header["expeditedDelivery"]).__name__},
        {"concept": "currency value", "path": "orders[].header.orderPrice", "example": repr(first_header["orderPrice"]), "Python type": type(first_header["orderPrice"]).__name__},
        {"concept": "percentage points", "path": "orders[].header.couponDiscount", "example": repr(first_header["couponDiscount"]), "Python type": type(first_header["couponDiscount"]).__name__},
        {"concept": "missing optional string", "path": "orders[].header.couponCode", "example": "empty string count=" + str(sum(order["header"].get("couponCode") == "" for order in json_orders)), "Python type": "str"},
    ]
)
show(json_formats, "JSON source-format evidence")

json_collections = {
    "customerProfiles[]": json_customers,
    "orders[].header": [order["header"] for order in json_orders],
    "orders[].shoppingCart[]": json_items,
    "orders[].delivery": json_deliveries,
    "productReviews[]": json_reviews,
}
json_missing_rows = []
for source_collection, records in json_collections.items():
    fields = sorted({field for record in records for field in record})
    for field in fields:
        absent_count = sum(field not in record for record in records)
        null_count = sum(record.get(field) is None for record in records if field in record)
        empty_string_count = sum(record.get(field) == "" for record in records if field in record)
        if absent_count or null_count or empty_string_count:
            json_missing_rows.append(
                {
                    "source_collection": source_collection,
                    "field": field,
                    "absent_key": absent_count,
                    "JSON null": null_count,
                    "empty_string": empty_string_count,
                }
            )
json_missing_profile = pd.DataFrame(json_missing_rows)
show(json_missing_profile, "JSON missing-value conventions (non-zero counts only)")



JSON collection profile
      source_collection                 grain candidate_key  rows  missing_key  unique_key  duplicate_key_groups  duplicate_extra_rows  exact_duplicate_groups  exact_duplicate_extra_rows
     customerProfiles[]  one customer profile    customerID   500            0         500                     0                     0                       0                           0
               orders[]      one source order       orderID  2818            0        2750                    68                    68                      68                          68
orders[].shoppingCart[] one source order item   orderItemID  8884            0        8666                   218                   218                     218                         218
      orders[].delivery   one source delivery    deliveryID  2818            0        2750                    68                    68                      68                          68
       productReviews[]     one source r

### 1.2 XML structure and profile


In [5]:
# EVIDENCE: SEC-1.2-XML-PROFILE
xml_tree = ET.parse(XML_PATH)
xml_root = xml_tree.getroot()

xml_orders = xml_root.findall("./Orders/Order")
xml_headers = [order.find("Header") for order in xml_orders]
xml_items = xml_root.findall("./Orders/Order/Shopping_Cart/Item")
xml_deliveries = xml_root.findall("./Orders/Order/Delivery")
xml_products = xml_root.findall("./ProductCatalogue/Product")
xml_reviews = xml_root.findall("./ProductReviews/Review")
xml_warehouses = list(xml_root.find("WarehouseDirectory"))

def element_records(elements):
    return [{child.tag: child.text or "" for child in element} for element in elements]

xml_header_records = element_records(xml_headers)
xml_item_records = element_records(xml_items)
xml_delivery_records = element_records(xml_deliveries)
xml_product_records = element_records(xml_products)
xml_review_records = element_records(xml_reviews)
xml_warehouse_records = element_records(xml_warehouses)

xml_profile = pd.DataFrame(
    [
        profile_records("/OperationsExport/Orders/Order/Header", "one source order", xml_header_records, "Order_ID"),
        profile_records("/OperationsExport/Orders/Order/Shopping_Cart/Item", "one source order item", xml_item_records, "Order_Item_ID"),
        profile_records("/OperationsExport/Orders/Order/Delivery", "one source delivery", xml_delivery_records, "Delivery_ID"),
        profile_records("/OperationsExport/ProductCatalogue/Product", "one product", xml_product_records, "Product_ID"),
        profile_records("/OperationsExport/ProductReviews/Review", "one source review", xml_review_records, "Review_ID"),
        profile_records("/OperationsExport/WarehouseDirectory/*", "one warehouse reference", xml_warehouse_records, "Name"),
    ]
)
show(xml_profile, "XML collection profile")

xml_structure = pd.DataFrame(
    [
        {"path": "/OperationsExport", "representation": "root element", "child elements": ", ".join(child.tag for child in xml_root)},
        {"path": "/OperationsExport/Orders/Order", "representation": "repeated element with Header, Shopping_Cart and Delivery", "child elements": ", ".join(child.tag for child in xml_orders[0])},
        {"path": "/OperationsExport/Orders/Order/Shopping_Cart/Item", "representation": "nested repeated element", "child elements": ", ".join(xml_item_records[0].keys())},
        {"path": "/OperationsExport/ProductCatalogue/Product", "representation": "repeated element", "child elements": ", ".join(xml_product_records[0].keys())},
        {"path": "/OperationsExport/ProductReviews/Review", "representation": "repeated element", "child elements": ", ".join(xml_review_records[0].keys())},
        {"path": "/OperationsExport/WarehouseDirectory/*", "representation": "source-specific reference collection", "child elements": ", ".join(xml_warehouse_records[0].keys())},
    ]
)
show(xml_structure, "XML nesting and fields")

first_xml_header = xml_header_records[0]
first_xml_delivery = xml_delivery_records[0]
xml_formats = pd.DataFrame(
    [
        {"concept": "timestamp", "path": ".../Header/Order_Timestamp", "example": repr(first_xml_header["Order_Timestamp"]), "XML representation": "text"},
        {"concept": "date", "path": ".../Delivery/Dispatch_Date", "example": repr(first_xml_delivery["Dispatch_Date"]), "XML representation": "text"},
        {"concept": "boolean", "path": ".../Header/Expedited_Delivery", "example": repr(first_xml_header["Expedited_Delivery"]), "XML representation": "Y/N text"},
        {"concept": "currency value", "path": ".../Header/Order_Price", "example": repr(first_xml_header["Order_Price"]), "XML representation": "currency-labelled text"},
        {"concept": "percentage", "path": ".../Header/Coupon_Discount", "example": repr(first_xml_header["Coupon_Discount"]), "XML representation": "percent-labelled text"},
        {"concept": "missing optional string", "path": ".../Header/Coupon_Code", "example": "empty element count=" + str(sum(record["Coupon_Code"] == "" for record in xml_header_records)), "XML representation": "empty element"},
    ]
)
show(xml_formats, "XML source-format evidence")

xml_collections = {
    "/OperationsExport/Orders/Order/Header": xml_header_records,
    "/OperationsExport/Orders/Order/Shopping_Cart/Item": xml_item_records,
    "/OperationsExport/Orders/Order/Delivery": xml_delivery_records,
    "/OperationsExport/ProductCatalogue/Product": xml_product_records,
    "/OperationsExport/ProductReviews/Review": xml_review_records,
}
xml_missing_rows = []
for source_collection, records in xml_collections.items():
    fields = sorted({field for record in records for field in record})
    for field in fields:
        absent_element_count = sum(field not in record for record in records)
        empty_element_count = sum(record.get(field) == "" for record in records)
        if absent_element_count or empty_element_count:
            xml_missing_rows.append(
                {
                    "source_collection": source_collection,
                    "field": field,
                    "absent_element": absent_element_count,
                    "empty_element": empty_element_count,
                }
            )
xml_missing_profile = pd.DataFrame(xml_missing_rows)
show(xml_missing_profile, "XML missing-value conventions (non-zero counts only)")



XML collection profile
                                source_collection                   grain candidate_key  rows  missing_key  unique_key  duplicate_key_groups  duplicate_extra_rows  exact_duplicate_groups  exact_duplicate_extra_rows
            /OperationsExport/Orders/Order/Header        one source order      Order_ID  2818            0        2750                    68                    68                      68                          68
/OperationsExport/Orders/Order/Shopping_Cart/Item   one source order item Order_Item_ID  8833            0        8622                   211                   211                     211                         211
          /OperationsExport/Orders/Order/Delivery     one source delivery   Delivery_ID  2818            0        2750                    68                    68                      68                          68
       /OperationsExport/ProductCatalogue/Product             one product    Product_ID  1000            0        10

### 1.3 Source comparison and assumptions


In [6]:
# EVIDENCE: SEC-1.3-SOURCE-COMPARISON
json_key_sets = {
    "orders": {record["orderID"] for record in [order["header"] for order in json_orders]},
    "order_items": {record["orderItemID"] for record in json_items},
    "deliveries": {record["deliveryID"] for record in json_deliveries},
    "product_reviews": {record["reviewID"] for record in json_reviews},
}
xml_key_sets = {
    "orders": {record["Order_ID"] for record in xml_header_records},
    "order_items": {record["Order_Item_ID"] for record in xml_item_records},
    "deliveries": {record["Delivery_ID"] for record in xml_delivery_records},
    "product_reviews": {record["Review_ID"] for record in xml_review_records},
}

overlap_rows = []
for entity in json_key_sets:
    json_keys = json_key_sets[entity]
    xml_keys = xml_key_sets[entity]
    overlap = json_keys & xml_keys
    overlap_rows.append(
        {
            "entity": entity,
            "JSON unique keys": len(json_keys),
            "XML unique keys": len(xml_keys),
            "cross-source overlap": len(overlap),
            "JSON only": len(json_keys - xml_keys),
            "XML only": len(xml_keys - json_keys),
            "union before field reconciliation": len(json_keys | xml_keys),
        }
    )
overlap_profile = pd.DataFrame(overlap_rows)
show(overlap_profile, "Cross-source key overlap")

combined_repeat_profile = pd.concat(
    [
        json_profile.assign(source="JSON"),
        xml_profile.assign(source="XML"),
    ],
    ignore_index=True,
)[
    ["source", "source_collection", "grain", "candidate_key", "rows", "missing_key", "unique_key", "duplicate_key_groups", "duplicate_extra_rows", "exact_duplicate_groups", "exact_duplicate_extra_rows"]
]
show(combined_repeat_profile, "Within-source repeat evidence")

source_coverage = pd.DataFrame(
    [
        {"target entity": "orders", "JSON evidence": "orders[].header", "XML evidence": "/OperationsExport/Orders/Order/Header", "coverage": "both"},
        {"target entity": "order_items", "JSON evidence": "orders[].shoppingCart[]", "XML evidence": "/OperationsExport/Orders/Order/Shopping_Cart/Item", "coverage": "both"},
        {"target entity": "customers", "JSON evidence": "customerProfiles[]", "XML evidence": "no full customer collection", "coverage": "JSON only"},
        {"target entity": "deliveries", "JSON evidence": "orders[].delivery", "XML evidence": "/OperationsExport/Orders/Order/Delivery", "coverage": "both"},
        {"target entity": "products", "JSON evidence": "product IDs only in related records", "XML evidence": "/OperationsExport/ProductCatalogue/Product", "coverage": "XML only for full entity"},
        {"target entity": "product_reviews", "JSON evidence": "productReviews[]", "XML evidence": "/OperationsExport/ProductReviews/Review", "coverage": "both"},
        {"target entity": "warehouse reference", "JSON evidence": "warehouse name in order header", "XML evidence": "/OperationsExport/WarehouseDirectory/*", "coverage": "source-specific helper"},
    ]
)
show(source_coverage, "Source coverage and source-specific collections")

key_relationships = pd.DataFrame(
    [
        {"entity / grain": "orders / one order", "candidate primary key": "order_id", "candidate foreign keys": "customer_id -> customers.customer_id", "source evidence": "header customer and order identifiers in both sources"},
        {"entity / grain": "order_items / one line item", "candidate primary key": "order_item_id", "candidate foreign keys": "order_id -> orders.order_id | product_id -> products.product_id", "source evidence": "nested shopping-cart identifiers in both sources"},
        {"entity / grain": "customers / one customer", "candidate primary key": "customer_id", "candidate foreign keys": "none", "source evidence": "customerProfiles[].customerID"},
        {"entity / grain": "deliveries / one delivery", "candidate primary key": "delivery_id", "candidate foreign keys": "order_id -> orders.order_id", "source evidence": "nested delivery identifiers in both sources"},
        {"entity / grain": "products / one product", "candidate primary key": "product_id", "candidate foreign keys": "none in the six-table target contract", "source evidence": "/OperationsExport/ProductCatalogue/Product/Product_ID"},
        {"entity / grain": "product_reviews / one review", "candidate primary key": "review_id", "candidate foreign keys": "order_id -> orders.order_id | order_item_id -> order_items.order_item_id | product_id -> products.product_id | customer_id -> customers.customer_id", "source evidence": "review identifiers in both sources"},
        {"entity / grain": "warehouse reference / one warehouse", "candidate primary key": "Name", "candidate foreign keys": "referenced by orders.nearest_warehouse", "source evidence": "/OperationsExport/WarehouseDirectory/*/Name"},
    ]
)
show(key_relationships, "Candidate keys and relationships to verify after reconciliation")

assumptions = pd.DataFrame(
    [
        {"assumption_id": "ASM-01", "decision before transformation": "Use the published table primary key as the stable business key at each target grain; do not invent identifiers."},
        {"assumption_id": "ASM-02", "decision before transformation": "Normalise comparable types and strings before comparing duplicates or cross-source overlap."},
        {"assumption_id": "ASM-03", "decision before transformation": "Retain one canonical row when all normalised non-missing values agree; record any disagreement as validation evidence rather than applying source precedence."},
        {"assumption_id": "ASM-04", "decision before transformation": "Preserve identifier leading zeros and source case; do not lower-case structured categories."},
        {"assumption_id": "ASM-05", "decision before transformation": "Parse JSON/XML structurally before applying regex to bounded narrative fields."},
        {"assumption_id": "ASM-06", "decision before transformation": "Use literal NaN only for prescribed missing string outputs; never substitute it into required numeric, boolean, primary-key or foreign-key fields."},
        {"assumption_id": "ASM-07", "decision before transformation": "Recompute line and order arithmetic in the published sequence and use reported source amounts only as validation evidence."},
        {"assumption_id": "ASM-08", "decision before transformation": "Flatten repeated items only into their target one-to-many table; do not flatten all entities into one wide table."},
        {"assumption_id": "ASM-09", "decision before transformation": "Treat the observed counts as profiling evidence only; derive every output and validation result from the current allocated files."},
    ]
)
show(assumptions, "Pre-transformation assumption register")

assert combined_repeat_profile["missing_key"].sum() == 0, "A candidate source key is missing"
duplicate_keys_are_exact = (
    combined_repeat_profile["duplicate_key_groups"]
    == combined_repeat_profile["exact_duplicate_groups"]
).all()
assert duplicate_keys_are_exact, "A repeated business key is not an exact source duplicate"
print(f"Repeated business-key groups are exact source duplicates: {duplicate_keys_are_exact}")
print("\nTask 1 source-profile checks: PASS")



Cross-source key overlap
         entity  JSON unique keys  XML unique keys  cross-source overlap  JSON only  XML only  union before field reconciliation
         orders              2750             2750                   500       2250      2250                               5000
    order_items              8666             8622                  1577       7089      7045                              15711
     deliveries              2750             2750                   500       2250      2250                               5000
product_reviews              3850             3850                   700       3150      3150                               7000

Within-source repeat evidence
source                                 source_collection                   grain candidate_key  rows  missing_key  unique_key  duplicate_key_groups  duplicate_extra_rows  exact_duplicate_groups  exact_duplicate_extra_rows
  JSON                                customerProfiles[]    one customer pro

## 2. Source-to-target mapping

The completed field-lineage artifact is
`Group050_source_to_target_mapping.csv` in the submission root. It preserves
the template's target rows and records applicable JSON/XML structural paths,
transformation or derivation, overlap/conflict handling and stable notebook
evidence for every target field. A source path may remain blank only when that
source is not applicable, as indicated by `source_format`.

The executable audit below checks template and dictionary alignment, applicable
path completeness, actual path existence in the parsed sources, evidence IDs,
placeholder absence, unique mapping IDs and UTF-8 encoding.


In [7]:
# EVIDENCE: SEC-2-MAPPING-AUDIT
import re

mapping_template = pd.read_csv(
    MAPPING_TEMPLATE_PATH, keep_default_na=False, dtype=str, encoding="utf-8-sig"
)
mapping = pd.read_csv(
    MAPPING_OUTPUT_PATH, keep_default_na=False, dtype=str, encoding="utf-8-sig"
)
data_dictionary = pd.read_csv(
    DATA_DICTIONARY_PATH, keep_default_na=False, dtype=str, encoding="utf-8-sig"
)

expected_columns = [
    "mapping_id",
    "output_table",
    "target_field",
    "source_format",
    "json_source_path",
    "xml_source_path",
    "transformation_or_derivation",
    "overlap_or_conflict_rule",
    "notebook_evidence",
]
prefilled_columns = ["mapping_id", "output_table", "target_field"]
required_text_columns = [
    "source_format",
    "transformation_or_derivation",
    "overlap_or_conflict_rule",
    "notebook_evidence",
]
completed_columns = [
    "source_format",
    "json_source_path",
    "xml_source_path",
    "transformation_or_derivation",
    "overlap_or_conflict_rule",
    "notebook_evidence",
]
allowed_source_formats = {"JSON", "XML", "both", "derived"}
defined_evidence_ids = {
    "SEC-1.1-JSON-PROFILE",
    "SEC-1.2-XML-PROFILE",
    "SEC-1.3-SOURCE-COMPARISON",
    "SEC-2-MAPPING-AUDIT",
}

audit_rows = []

def add_mapping_check(check_id, check, observed, passed):
    audit_rows.append(
        {
            "check_id": check_id,
            "check": check,
            "observed": observed,
            "status": "PASS" if passed else "FAIL",
        }
    )

add_mapping_check(
    "MAP-AUDIT-01",
    "Required mapping filename and file presence",
    MAPPING_OUTPUT_PATH.name,
    MAPPING_OUTPUT_PATH.is_file()
    and MAPPING_OUTPUT_PATH.name == f"{GROUP_ID}_source_to_target_mapping.csv",
)
add_mapping_check(
    "MAP-AUDIT-02",
    "Exact nine-column contract",
    list(mapping.columns),
    list(mapping.columns) == expected_columns,
)
template_alignment = (
    len(mapping) == len(mapping_template)
    and mapping[prefilled_columns].equals(mapping_template[prefilled_columns])
)
add_mapping_check(
    "MAP-AUDIT-03",
    "All pre-filled template rows preserved in order",
    f"mapping={len(mapping)}, template={len(mapping_template)}",
    template_alignment,
)
dictionary_targets = data_dictionary[["output_table", "field_name"]].rename(
    columns={"field_name": "target_field"}
)
dictionary_alignment = (
    len(mapping) == len(data_dictionary)
    and mapping[["output_table", "target_field"]].equals(dictionary_targets)
)
add_mapping_check(
    "MAP-AUDIT-04",
    "Mapping targets match the public data dictionary",
    f"mapping={len(mapping)}, dictionary={len(data_dictionary)}",
    dictionary_alignment,
)
mapping_id_ok = (
    mapping["mapping_id"].is_unique
    and mapping["mapping_id"].str.fullmatch(r"MAP-[a-z_]+-\d{2}").all()
)
add_mapping_check(
    "MAP-AUDIT-05",
    "Stable and unique MAP IDs",
    f"unique={mapping['mapping_id'].nunique()}",
    mapping_id_ok,
)
source_format_ok = set(mapping["source_format"]) <= allowed_source_formats
add_mapping_check(
    "MAP-AUDIT-06",
    "Allowed source_format values only",
    sorted(mapping["source_format"].unique()),
    source_format_ok,
)
blank_required_text = [
    (row.mapping_id, column)
    for row in mapping.itertuples(index=False)
    for column in required_text_columns
    if not str(getattr(row, column)).strip()
]
add_mapping_check(
    "MAP-AUDIT-07",
    "All non-path mapping fields completed",
    f"blank cells={len(blank_required_text)}",
    not blank_required_text,
)
applicable_path_missing = []
for row in mapping.itertuples(index=False):
    json_path_present = bool(row.json_source_path.strip())
    xml_path_present = bool(row.xml_source_path.strip())
    if row.source_format in {"JSON", "both"} and not json_path_present:
        applicable_path_missing.append((row.mapping_id, "json_source_path"))
    if row.source_format in {"XML", "both"} and not xml_path_present:
        applicable_path_missing.append((row.mapping_id, "xml_source_path"))
    if row.source_format == "derived" and not (json_path_present or xml_path_present):
        applicable_path_missing.append((row.mapping_id, "derived source path"))
add_mapping_check(
    "MAP-AUDIT-08",
    "Every applicable JSON/XML path is present",
    f"missing applicable paths={len(applicable_path_missing)}",
    not applicable_path_missing,
)

json_actual_paths = set()
for prefix, records in json_collections.items():
    for record in records:
        json_actual_paths.update(f"{prefix}.{field}" for field in record)

xml_actual_paths = set()
def collect_xml_paths(element, parent_path):
    for child in element:
        child_path = f"{parent_path}/{child.tag}"
        xml_actual_paths.add(child_path)
        collect_xml_paths(child, child_path)

collect_xml_paths(xml_root, f"/{xml_root.tag}")

def split_mapping_paths(value):
    return [part.strip() for part in value.split("|") if part.strip()]

invalid_path_references = []
for row in mapping.itertuples(index=False):
    for path in split_mapping_paths(row.json_source_path):
        if path not in json_actual_paths:
            invalid_path_references.append((row.mapping_id, "JSON", path))
    for path in split_mapping_paths(row.xml_source_path):
        if path not in xml_actual_paths:
            invalid_path_references.append((row.mapping_id, "XML", path))
add_mapping_check(
    "MAP-AUDIT-09",
    "Every listed structural path exists in the parsed sources",
    f"invalid path references={len(invalid_path_references)}",
    not invalid_path_references,
)
evidence_tokens = {
    token.strip()
    for value in mapping["notebook_evidence"]
    for token in value.split("|")
    if token.strip()
}
unknown_evidence_ids = sorted(evidence_tokens - defined_evidence_ids)
all_rows_cite_mapping_audit = mapping["notebook_evidence"].str.contains(
    "SEC-2-MAPPING-AUDIT", regex=False
).all()
add_mapping_check(
    "MAP-AUDIT-10",
    "Evidence IDs are defined and every row cites this audit",
    f"unknown={unknown_evidence_ids}, all cite audit={all_rows_cite_mapping_audit}",
    not unknown_evidence_ids and all_rows_cite_mapping_audit,
)
placeholder_pattern = re.compile(r"\b(?:replace|todo|tbd|example)\b", re.IGNORECASE)
placeholder_rows = mapping[completed_columns].apply(
    lambda column: column.map(lambda value: bool(placeholder_pattern.search(value)))
).any(axis=1)
add_mapping_check(
    "MAP-AUDIT-11",
    "No template placeholders remain in completed fields",
    f"placeholder rows={int(placeholder_rows.sum())}",
    not placeholder_rows.any(),
)
utf8_bom_present = MAPPING_OUTPUT_PATH.read_bytes().startswith(b"\xef\xbb\xbf")
add_mapping_check(
    "MAP-AUDIT-12",
    "CSV is UTF-8 with BOM for portable spreadsheet display",
    f"UTF-8 BOM={utf8_bom_present}",
    utf8_bom_present,
)

mapping_audit = pd.DataFrame(audit_rows)
show(mapping_audit, "Task 1 mapping audit register")
mapping_coverage = (
    mapping.groupby(["output_table", "source_format"], sort=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
show(mapping_coverage, "Mapping coverage by target table and source format")
failed_mapping_checks = mapping_audit[mapping_audit["status"] != "PASS"]
assert failed_mapping_checks.empty, failed_mapping_checks.to_string(index=False)
print(f"\nFINAL_TASK1_MAPPING_AUDIT_PASS True; rows={len(mapping)}")



Task 1 mapping audit register
    check_id                                                     check                                                                                                                                                              observed status
MAP-AUDIT-01               Required mapping filename and file presence                                                                                                                                 Group050_source_to_target_mapping.csv   PASS
MAP-AUDIT-02                                Exact nine-column contract [mapping_id, output_table, target_field, source_format, json_source_path, xml_source_path, transformation_or_derivation, overlap_or_conflict_rule, notebook_evidence]   PASS
MAP-AUDIT-03           All pre-filled template rows preserved in order                                                                                                                                             mapping=111, template=111 

## Task 1 stage boundary

Task 1 ends above. The remaining sections are retained as official template
scaffolding and are intentionally not implemented or executed in this Task 1
checkpoint. They will be completed only in later stage versions.

## 3. Text and regex functions


### 3.1 Cleaning and extraction implementation


### 3.2 Public and student-designed tests


## 4. Build the six standardised relational tables

Show the transformation and row-flow evidence for each table. Keep helper
columns inside the workflow; export only fields in the public data dictionary.


### 4.1 `orders`


### 4.2 `order_items`


### 4.3 `customers`


### 4.4 `deliveries`


### 4.5 `products`


In [8]:
# EVIDENCE: SEC-4.5-PRODUCTS
# TASK SCOPE: Build only the Task 2 standardised products table.
# This cell does not export a CSV or implement the Task 3 deliverable module.

from datetime import datetime
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import html
import re
import unicodedata

import pandas as pd


def _normalise_required_product_text(value, field_name):
    """
    Usage:
        _normalise_required_product_text(raw_value, "Product_ID")

    Reason:
        Required product identifiers and structured categories must be trimmed
        and NFC-normalised while preserving leading zeros and source case.

    Expected result:
        A non-empty string. Missing or empty input raises ValueError.
    """
    if value is None:
        raise ValueError(f"{field_name}: required text is missing")

    result = unicodedata.normalize("NFC", str(value)).strip()

    if not result:
        raise ValueError(f"{field_name}: required text is empty")

    return result


def _parse_product_currency(value, field_name):
    """
    Usage:
        _parse_product_currency("AUD 2,686.14", "Unit_Price")

    Reason:
        XML prices contain an AUD label and thousands separators that must be
        removed before numeric conversion.

    Expected result:
        A float rounded to two decimal places, such as 2686.14.
    """
    text = _normalise_required_product_text(value, field_name)
    numeric_text = re.sub(r"(?i)\bAUD\b", "", text)
    numeric_text = numeric_text.replace(",", "").strip()

    try:
        amount = Decimal(numeric_text)
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid currency value {value!r}"
        ) from exc

    return float(
        amount.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
    )


def _parse_product_integer(value, field_name):
    """
    Usage:
        _parse_product_integer("24", "Warranty_Months")

    Reason:
        Integer product attributes must be converted without silently rounding
        non-integral values.

    Expected result:
        A Python int. Invalid or non-integral input raises ValueError.
    """
    text = _normalise_required_product_text(value, field_name)

    try:
        number = Decimal(text)
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid integer value {value!r}"
        ) from exc

    if number != number.to_integral_value():
        raise ValueError(
            f"{field_name}: expected an integer, got {value!r}"
        )

    return int(number)


def _parse_product_number(value, field_name):
    """
    Usage:
        _parse_product_number("1.551", "Weight_Kg")

    Reason:
        Numeric measurements must be converted from XML text while retaining
        their published precision.

    Expected result:
        A Python float, such as 1.551.
    """
    text = _normalise_required_product_text(value, field_name)

    try:
        return float(Decimal(text))
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid numeric value {value!r}"
        ) from exc


def _parse_product_date(value, field_name):
    """
    Usage:
        _parse_product_date("26/06/2014", "Launch_Date")

    Reason:
        XML Product launch dates use DD/MM/YYYY, while the target requires
        YYYY-MM-DD.

    Expected result:
        A date string such as "2014-06-26".
    """
    text = _normalise_required_product_text(value, field_name)

    try:
        return datetime.strptime(text, "%d/%m/%Y").strftime("%Y-%m-%d")
    except ValueError as exc:
        raise ValueError(
            f"{field_name}: invalid DD/MM/YYYY date {value!r}"
        ) from exc


def _parse_product_boolean(value, field_name):
    """
    Usage:
        _parse_product_boolean("Y", "Active_Flag")

    Reason:
        XML represents Product booleans as Y/N; the target requires True/False.

    Expected result:
        True for Y and False for N. Other values raise ValueError.
    """
    text = _normalise_required_product_text(
        value, field_name
    ).upper()

    boolean_mapping = {
        "Y": True,
        "N": False,
    }

    if text not in boolean_mapping:
        raise ValueError(
            f"{field_name}: expected Y or N, got {value!r}"
        )

    return boolean_mapping[text]


def _remove_product_description_emoji(text):
    """
    Usage:
        _remove_product_description_emoji(normalised_text)

    Reason:
        The published narrative-cleaning sequence removes emoji. This private
        helper is limited to the Product table and is not the assessed Task 3
        function.

    Expected result:
        The same string with common emoji code-point ranges removed.
    """
    emoji_ranges = (
        (0x1F000, 0x1FAFF),
        (0x2600, 0x27BF),
        (0x2300, 0x23FF),
        (0x2B00, 0x2BFF),
        (0xFE00, 0xFE0F),
        (0x1F1E6, 0x1F1FF),
    )

    return "".join(
        character
        for character in text
        if character not in {"\u200d", "\u20e3"}
        and not any(
            start <= ord(character) <= end
            for start, end in emoji_ranges
        )
    )


def _clean_product_description_for_task2(value):
    """
    Usage:
        _clean_product_description_for_task2(raw_product_description)

    Reason:
        Task 2 requires product_description_clean now, but the shared assessed
        clean_narrative_text function belongs to Task 3 and may not exist yet.

        If clean_narrative_text has already been loaded by the final combined
        workflow, this helper delegates to it. Otherwise, it applies the
        published cleaning sequence privately for the Product table only.

    Expected result:
        Lower-case cleaned text, or the literal string "NaN" when no readable
        text remains.
    """
    shared_cleaner = globals().get("clean_narrative_text")

    if callable(shared_cleaner):
        result = shared_cleaner(value)

        if not isinstance(result, str) or result == "":
            raise ValueError(
                "clean_narrative_text must return non-empty text or "
                "the literal string 'NaN'."
            )

        return result

    if value is None:
        return "NaN"

    # 1. Decode HTML entities and apply Unicode NFC normalisation.
    text = html.unescape(str(value))
    text = unicodedata.normalize("NFC", text)

    # 2. Remove HTML/XML-like tags while preserving readable content.
    text = re.sub(r"<[^>]*>", " ", text)

    # 3. Remove fixed and parameterised published markers.
    text = re.sub(
        r"\[(?:SYSTEM|CATALOGUE|VERIFIED_PURCHASE)\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\[SOURCE:\s*[^\]]*\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\[RATING:\s*[0-5]\s*/\s*5\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"(?<![\w-])(?:#verified-buyer|@store_support)(?![\w-])",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # 4. Remove URLs.
    text = re.sub(
        r"(?i)\b(?:https?://|www\.)\S+",
        " ",
        text,
    )

    # 5. Remove emoji.
    text = _remove_product_description_emoji(text)

    # 6. Remove a complete review reference wrapper if one occurs.
    text = re.sub(
        (
            r"(?i)(?<![A-Z0-9])"
            r"Reference:\s*(?:HORD|CORD)\d{6}"
            r"\s*[|,;/]\s*"
            r"SKU:\s*SKU-[A-Z0-9]+"
            r"(?![A-Z0-9-])"
        ),
        " ",
        text,
    )

    # 7. Remove PROMO: together with a valid promotion code.
    text = re.sub(
        (
            r"(?i)(?<![A-Z0-9])"
            r"PROMO:\s*B[1-5]SAVE-\d{2}"
            r"(?![A-Z0-9-])"
        ),
        " ",
        text,
    )

    # 8. Collapse whitespace, trim and lower-case the cleaned narrative.
    text = re.sub(r"\s+", " ", text).strip().lower()

    # 9. Use the literal missing-string sentinel when no text remains.
    return text if text else "NaN"


def _standardise_product_record(record):
    """
    Usage:
        _standardise_product_record(one_xml_product_record)

    Reason:
        Converts one structured XML Product record to the exact target fields.
        XML structure has already been parsed in Task 1.

    Expected result:
        One dictionary containing exactly the 21 Product target fields.
    """
    return {
        "product_id": _normalise_required_product_text(
            record.get("Product_ID"),
            "Product_ID",
        ),
        "product_name": _normalise_required_product_text(
            record.get("Product_Name"),
            "Product_Name",
        ),
        "category": _normalise_required_product_text(
            record.get("Category"),
            "Category",
        ),
        "brand": _normalise_required_product_text(
            record.get("Brand"),
            "Brand",
        ),
        "unit_price": _parse_product_currency(
            record.get("Unit_Price"),
            "Unit_Price",
        ),
        "unit_cost": _parse_product_currency(
            record.get("Unit_Cost"),
            "Unit_Cost",
        ),
        "launch_year": _parse_product_integer(
            record.get("Launch_Year"),
            "Launch_Year",
        ),
        "warranty_months": _parse_product_integer(
            record.get("Warranty_Months"),
            "Warranty_Months",
        ),
        "weight_kg": _parse_product_number(
            record.get("Weight_Kg"),
            "Weight_Kg",
        ),
        "product_sku": _normalise_required_product_text(
            record.get("Product_Sku"),
            "Product_Sku",
        ),
        "subcategory": _normalise_required_product_text(
            record.get("Subcategory"),
            "Subcategory",
        ),
        "model_family": _normalise_required_product_text(
            record.get("Model_Family"),
            "Model_Family",
        ),
        "colour": _normalise_required_product_text(
            record.get("Colour"),
            "Colour",
        ),
        "supplier_id": _normalise_required_product_text(
            record.get("Supplier_ID"),
            "Supplier_ID",
        ),
        "supplier_country": _normalise_required_product_text(
            record.get("Supplier_Country"),
            "Supplier_Country",
        ),
        "launch_date": _parse_product_date(
            record.get("Launch_Date"),
            "Launch_Date",
        ),
        "tax_category": _normalise_required_product_text(
            record.get("Tax_Category"),
            "Tax_Category",
        ),
        "package_type": _normalise_required_product_text(
            record.get("Package_Type"),
            "Package_Type",
        ),
        "recyclable_packaging": _parse_product_boolean(
            record.get("Recyclable_Packaging"),
            "Recyclable_Packaging",
        ),
        "active_flag": _parse_product_boolean(
            record.get("Active_Flag"),
            "Active_Flag",
        ),
        "product_description_clean": (
            _clean_product_description_for_task2(
                record.get("Product_Description")
            )
        ),
    }


def _reconcile_product_candidates(frame):
    """
    Usage:
        products, conflicts = _reconcile_product_candidates(candidates)

    Reason:
        Product rows must be reconciled by product_id after normalisation.
        Conflicting non-missing values must be recorded rather than resolved
        using arbitrary row or source precedence.

    Expected result:
        One canonical row per product_id and a separate conflict DataFrame.
    """
    canonical_rows = []
    conflict_rows = []

    for product_id, group in frame.groupby(
        "product_id",
        sort=True,
        dropna=False,
    ):
        canonical = {"product_id": product_id}

        for column in frame.columns:
            if column == "product_id":
                continue

            non_missing_values = [
                value
                for value in group[column].tolist()
                if not pd.isna(value)
            ]

            # Remove duplicate values while preserving deterministic order.
            distinct_values = list(dict.fromkeys(non_missing_values))

            if len(distinct_values) > 1:
                conflict_rows.append(
                    {
                        "product_id": product_id,
                        "field": column,
                        "normalised_values": distinct_values,
                    }
                )

            canonical[column] = (
                distinct_values[0]
                if distinct_values
                else None
            )

        canonical_rows.append(canonical)

    canonical_frame = pd.DataFrame(
        canonical_rows,
        columns=frame.columns,
    )
    conflict_frame = pd.DataFrame(
        conflict_rows,
        columns=[
            "product_id",
            "field",
            "normalised_values",
        ],
    )

    return canonical_frame, conflict_frame


# Read the required Product schema and field order from the public dictionary.
# This avoids maintaining a second manual output schema.
product_columns = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("products")
    ]
    .assign(
        _position=lambda frame: frame["position"].astype(int)
    )
    .sort_values("_position")["field_name"]
    .tolist()
)

# xml_product_records was created using the structured XML parser in Task 1.
product_candidates = pd.DataFrame(
    [
        _standardise_product_record(record)
        for record in xml_product_records
    ],
    columns=product_columns,
)

# Reconcile by the published Product primary key.
products, product_reconciliation_conflicts = (
    _reconcile_product_candidates(product_candidates)
)

# Do not silently select a value when a conflict is found.
if not product_reconciliation_conflicts.empty:
    raise ValueError(
        "Conflicting non-missing normalised Product values were found:\n"
        + product_reconciliation_conflicts.to_string(index=False)
    )

# Keep only the required fields and make row ordering deterministic.
products = (
    products.loc[:, product_columns]
    .sort_values("product_id", kind="stable")
    .reset_index(drop=True)
)

# Immediate Task 2 transformation guards.
# These do not replace the later Task 4 validation register.
if list(products.columns) != product_columns:
    raise AssertionError(
        "Product columns do not match the public data dictionary."
    )

if products["product_id"].isna().any():
    raise AssertionError(
        "products.product_id contains missing values."
    )

if not products["product_id"].is_unique:
    raise AssertionError(
        "products.product_id is not unique."
    )

if products.isna().any().any():
    missing_counts = products.isna().sum()
    missing_counts = missing_counts[missing_counts.gt(0)]

    raise AssertionError(
        "Products contains prohibited missing values:\n"
        + missing_counts.to_string()
    )

product_string_columns = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("products")
        & data_dictionary["data_type"].eq("string"),
        "field_name",
    ]
    .tolist()
)

blank_string_counts = {
    column: int(
        products[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    for column in product_string_columns
}
blank_string_counts = {
    column: count
    for column, count in blank_string_counts.items()
    if count > 0
}

if blank_string_counts:
    raise AssertionError(
        "Products contains empty required strings: "
        f"{blank_string_counts}"
    )

# Display Task 2 row-flow evidence without hard-coding expected row counts.
product_row_flow = pd.DataFrame(
    [
        {
            "stage": "structured XML product records",
            "rows": len(xml_product_records),
        },
        {
            "stage": "normalised Product candidates",
            "rows": len(product_candidates),
        },
        {
            "stage": "canonical products",
            "rows": len(products),
        },
    ]
)

show(
    product_row_flow,
    "Product transformation row flow",
)
show(
    products.head(),
    "Standardised products preview",
)

cleaner_source = (
    "shared clean_narrative_text"
    if callable(globals().get("clean_narrative_text"))
    else "private Task 2 Product fallback"
)

print(
    "\nProduct table ready:"
    f" rows={len(products)},"
    f" columns={len(products.columns)},"
    f" reconciliation_conflicts="
    f"{len(product_reconciliation_conflicts)},"
    f" cleaner={cleaner_source}"
)


Product transformation row flow
                         stage  rows
structured XML product records  1000
 normalised Product candidates  1000
            canonical products  1000

Standardised products preview
product_id     product_name           category  brand  unit_price  unit_cost  launch_year  warranty_months  weight_kg  product_sku         subcategory model_family   colour supplier_id supplier_country launch_date tax_category package_type  recyclable_packaging  active_flag                                                                                                                        product_description_clean
   PRD0001 Candle Bloom 100             Laptop Candle     2686.14    1498.92         2014               24      1.551 SKU-CAN00001           Ultrabook          Arc Graphite      SUP001        Australia  2014-06-26 GST_STANDARD Recycled box                  True         True             ultrabook designed for portable document work and video meetings, in the arc fami

### 4.6 `product_reviews`


In [9]:
# EVIDENCE: SEC-4.6-PRODUCT-REVIEWS
# TASK SCOPE: Build only the Task 2 standardised product_reviews table.
# This cell does not export CSV files or write the Task 4 validation register.

from datetime import datetime
from decimal import Decimal, InvalidOperation
import html
import re
import unicodedata

import pandas as pd


# ---------------------------------------------------------------------------
# 1. Task 1 prerequisite check
# ---------------------------------------------------------------------------

# Usage:
#   Run this Product Review cell after all Task 1 cells and the Product cell.
# Reason:
#   Product Reviews reuses the structured JSON/XML objects and dictionary
#   produced by Task 1 instead of parsing the source documents again.
# Expected result:
#   No exception. If an object is missing, the message identifies what must
#   be run first.
required_product_review_inputs = [
    "data_dictionary",
    "json_reviews",
    "xml_review_records",
    "show",
]

missing_product_review_inputs = [
    name
    for name in required_product_review_inputs
    if name not in globals()
]

if missing_product_review_inputs:
    raise RuntimeError(
        "Run all Task 1 cells before the Product Review cell. "
        f"Missing objects: {missing_product_review_inputs}"
    )


# ---------------------------------------------------------------------------
# 2. General Product Review standardisation helpers
# ---------------------------------------------------------------------------

def _normalise_required_review_text(value, field_name):
    """
    Usage:
        _normalise_required_review_text(raw_value, "Review_ID")

    Reason:
        Required identifiers and structured review attributes must be trimmed
        and Unicode NFC-normalised without changing their source case or
        removing identifier leading zeros.

    Expected result:
        A non-empty string. Missing or empty input raises ValueError.
    """
    if value is None:
        raise ValueError(f"{field_name}: required text is missing")

    result = unicodedata.normalize("NFC", str(value)).strip()

    if not result:
        raise ValueError(f"{field_name}: required text is empty")

    return result


def _parse_required_review_integer(value, field_name):
    """
    Usage:
        _parse_required_review_integer("5", "Rating")

    Reason:
        Rating and helpful-vote values are represented as numbers in JSON but
        text in XML. Both sources must produce the same target type.

    Expected result:
        A Python int. Invalid or non-integral values raise ValueError.
    """
    text = _normalise_required_review_text(value, field_name)

    try:
        number = Decimal(text)
    except InvalidOperation as exc:
        raise ValueError(
            f"{field_name}: invalid integer value {value!r}"
        ) from exc

    if number != number.to_integral_value():
        raise ValueError(
            f"{field_name}: expected an integer, got {value!r}"
        )

    return int(number)


def _parse_review_timestamp(value, field_name, source_format):
    """
    Usage:
        _parse_review_timestamp(value, "Review_Timestamp", "XML")

    Reason:
        JSON timestamps use YYYY-MM-DD HH:MM:SS, while XML timestamps use
        DD/MM/YYYY HH:MM:SS. The target requires YYYY-MM-DD HH:MM:SS.

    Expected result:
        A standardised timestamp string such as "2018-06-16 18:58:00".
    """
    text = _normalise_required_review_text(value, field_name)

    source_patterns = {
        "JSON": "%Y-%m-%d %H:%M:%S",
        "XML": "%d/%m/%Y %H:%M:%S",
    }

    if source_format not in source_patterns:
        raise ValueError(
            f"Unsupported review source format: {source_format!r}"
        )

    try:
        parsed = datetime.strptime(
            text,
            source_patterns[source_format],
        )
    except ValueError as exc:
        raise ValueError(
            f"{field_name}: invalid {source_format} timestamp {value!r}"
        ) from exc

    return parsed.strftime("%Y-%m-%d %H:%M:%S")


def _parse_review_boolean(value, field_name, source_format):
    """
    Usage:
        _parse_review_boolean("Y", "Verified_Purchase", "XML")

    Reason:
        JSON uses real booleans, while XML uses Y/N. The standardised output
        must contain Python True or False.

    Expected result:
        A Python bool. Unsupported values raise ValueError.
    """
    if source_format == "JSON":
        if type(value) is not bool:
            raise ValueError(
                f"{field_name}: expected a JSON boolean, got {value!r}"
            )

        return value

    if source_format == "XML":
        text = _normalise_required_review_text(
            value,
            field_name,
        ).upper()

        xml_boolean_mapping = {
            "Y": True,
            "N": False,
        }

        if text not in xml_boolean_mapping:
            raise ValueError(
                f"{field_name}: expected Y or N, got {value!r}"
            )

        return xml_boolean_mapping[text]

    raise ValueError(
        f"Unsupported review source format: {source_format!r}"
    )


# ---------------------------------------------------------------------------
# 3. Private Task 2 text fallbacks
#
# These private helpers let Task 2 Product Reviews run independently.
# They do not define or overwrite the six assessed Task 3 function names.
# If a teammate's shared function is later loaded, this cell automatically
# calls that shared function instead.
# ---------------------------------------------------------------------------

def _call_shared_function_or_fallback(
    shared_function_name,
    fallback_function,
    value,
):
    """
    Usage:
        _call_shared_function_or_fallback(
            "extract_order_reference",
            _fallback_extract_order_reference,
            raw_review,
        )

    Reason:
        Task 2 needs derived review fields before the shared Task 3 deliverables
        are integrated. This preserves independence without redefining another
        teammate's public function.

    Expected result:
        The shared function result when it is available; otherwise the private
        Task 2 fallback result.
    """
    shared_function = globals().get(shared_function_name)

    if callable(shared_function):
        return shared_function(value)

    return fallback_function(value)


def _remove_task2_review_emoji(text):
    """
    Usage:
        _remove_task2_review_emoji(normalised_review_text)

    Reason:
        The published cleaning sequence removes emoji from cleaned narrative
        fields.

    Expected result:
        The input string with common emoji code-point ranges removed.
    """
    emoji_ranges = (
        (0x1F000, 0x1FAFF),
        (0x2600, 0x27BF),
        (0x2300, 0x23FF),
        (0x2B00, 0x2BFF),
        (0xFE00, 0xFE0F),
        (0x1F1E6, 0x1F1FF),
    )

    return "".join(
        character
        for character in text
        if character not in {"\u200d", "\u20e3"}
        and not any(
            start <= ord(character) <= end
            for start, end in emoji_ranges
        )
    )


def _fallback_clean_review_text(value):
    """
    Usage:
        _fallback_clean_review_text(raw_review_text)

    Reason:
        Produces review_body_clean during Task 2 when the teammate-owned
        clean_narrative_text function has not yet been integrated.

    Expected result:
        Lower-case multilingual cleaned text, or the literal string "NaN".
    """
    if value is None:
        return "NaN"

    # Decode HTML entities and apply Unicode NFC normalisation.
    text = html.unescape(str(value))
    text = unicodedata.normalize("NFC", text)

    # Remove HTML/XML-like tags while retaining readable content.
    text = re.sub(r"<[^>]*>", " ", text)

    # Remove the published fixed markers.
    text = re.sub(
        r"\[(?:SYSTEM|CATALOGUE|VERIFIED_PURCHASE)\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # Remove the published parameterised markers.
    text = re.sub(
        r"\[SOURCE:\s*[^\]]*\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\[RATING:\s*[0-5]\s*/\s*5\]",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # Remove the two published social markers.
    text = re.sub(
        r"(?<![\w-])(?:#verified-buyer|@store_support)(?![\w-])",
        " ",
        text,
        flags=re.IGNORECASE,
    )

    # Remove URLs.
    text = re.sub(
        r"(?i)\b(?:https?://|www\.)\S+",
        " ",
        text,
    )

    # Remove emoji.
    text = _remove_task2_review_emoji(text)

    # Remove a complete review reference wrapper.
    text = re.sub(
        (
            r"(?i)(?<![A-Z0-9_])"
            r"Reference:\s*(?:HORD|CORD)\d{6}"
            r"\s*[|,;/]\s*"
            r"SKU:\s*SKU-[A-Z0-9]+"
            r"(?![A-Z0-9-])"
        ),
        " ",
        text,
    )

    # Remove PROMO: together with a valid promotion code.
    text = re.sub(
        (
            r"(?i)(?<![A-Z0-9_])"
            r"PROMO:\s*B[1-5]SAVE-\d{2}"
            r"(?![A-Z0-9-])"
        ),
        " ",
        text,
    )

    # Collapse spaces, tabs and line breaks; trim and lower-case.
    text = re.sub(r"\s+", " ", text).strip().lower()

    return text if text else "NaN"


def _fallback_extract_order_reference(value):
    """
    Usage:
        _fallback_extract_order_reference(raw_review_text)

    Reason:
        extracted_order_reference must be obtained from the raw review before
        narrative cleaning removes the reference wrapper.

    Expected result:
        An upper-case HORD/CORD reference followed by exactly six digits, or
        the literal string "NaN".
    """
    if value is None:
        return "NaN"

    match = re.search(
        (
            r"(?i)(?<![A-Z0-9_])"
            r"(?:HORD|CORD)\d{6}"
            r"(?![A-Z0-9_])"
        ),
        str(value),
    )

    return match.group(0).upper() if match else "NaN"


def _fallback_extract_product_sku(value):
    """
    Usage:
        _fallback_extract_product_sku(raw_review_text)

    Reason:
        extracted_product_sku must be obtained from the raw review before the
        reference wrapper is removed.

    Expected result:
        An upper-case SKU followed by ASCII letters/digits, or "NaN".
        Malformed extensions such as SKU-ABC123-extra are rejected.
    """
    if value is None:
        return "NaN"

    match = re.search(
        (
            r"(?i)(?<![A-Z0-9_-])"
            r"SKU-[A-Z0-9]+"
            r"(?![A-Z0-9-])"
        ),
        str(value),
    )

    return match.group(0).upper() if match else "NaN"


def _fallback_build_latin_analysis(value):
    """
    Usage:
        _fallback_build_latin_analysis(review_body_clean)

    Reason:
        The Latin-analysis field must be derived from cleaned multilingual
        review text, not from the noisy raw review.

    Expected result:
        Text retaining Latin-script letters, European diacritics, applicable
        digits and punctuation. Returns literal "NaN" if no Latin letter
        remains.
    """
    if value is None or value == "NaN":
        return "NaN"

    text = unicodedata.normalize("NFC", str(value))
    output_characters = []
    contains_latin_letter = False
    previous_character_was_latin = False

    for character in text:
        category = unicodedata.category(character)
        unicode_name = unicodedata.name(character, "")

        if character.isalpha():
            if "LATIN" in unicode_name:
                output_characters.append(character)
                contains_latin_letter = True
                previous_character_was_latin = True
            else:
                # A space prevents words on either side of removed script
                # characters from being joined accidentally.
                output_characters.append(" ")
                previous_character_was_latin = False

        elif category.startswith("M"):
            # Retain a combining mark only when it belongs to a retained
            # Latin-script letter.
            if previous_character_was_latin:
                output_characters.append(character)

        else:
            output_characters.append(character)
            previous_character_was_latin = False

    result = re.sub(
        r"\s+",
        " ",
        "".join(output_characters),
    ).strip()

    if not contains_latin_letter or not result:
        return "NaN"

    return result


def _fallback_contains_non_latin_script(value):
    """
    Usage:
        _fallback_contains_non_latin_script(review_body_clean)

    Reason:
        The indicator must detect letters outside the Latin script without
        treating every non-ASCII character as non-Latin.

    Expected result:
        True if a cleaned review contains a non-Latin letter; otherwise False.
    """
    if value is None or value == "NaN":
        return False

    text = unicodedata.normalize("NFC", str(value))

    return any(
        character.isalpha()
        and "LATIN" not in unicodedata.name(character, "")
        for character in text
    )


# ---------------------------------------------------------------------------
# 4. Convert one structured JSON/XML review to target fields
# ---------------------------------------------------------------------------

def _standardise_product_review_record(record, source_format):
    """
    Usage:
        _standardise_product_review_record(record, "JSON")
        _standardise_product_review_record(record, "XML")

    Reason:
        Maps the two source-specific representations to the same 21-field
        Product Review target schema.

    Expected result:
        One normalised dictionary at review grain with no helper fields.
    """
    source_field_maps = {
        "JSON": {
            "review_id": "reviewID",
            "order_id": "orderID",
            "order_item_id": "orderItemID",
            "product_id": "productID",
            "customer_id": "customerID",
            "review_timestamp": "reviewTimestamp",
            "language_code": "languageCode",
            "rating": "rating",
            "review_title": "reviewTitle",
            "review_text": "reviewText",
            "verified_purchase": "verifiedPurchase",
            "helpful_votes": "helpfulVotes",
            "delivery_experience": "deliveryExperience",
            "value_experience": "valueExperience",
            "writing_style": "writingStyle",
        },
        "XML": {
            "review_id": "Review_ID",
            "order_id": "Order_ID",
            "order_item_id": "Order_Item_ID",
            "product_id": "Product_ID",
            "customer_id": "Customer_ID",
            "review_timestamp": "Review_Timestamp",
            "language_code": "Language_Code",
            "rating": "Rating",
            "review_title": "Review_Title",
            "review_text": "Review_Text",
            "verified_purchase": "Verified_Purchase",
            "helpful_votes": "Helpful_Votes",
            "delivery_experience": "Delivery_Experience",
            "value_experience": "Value_Experience",
            "writing_style": "Writing_Style",
        },
    }

    if source_format not in source_field_maps:
        raise ValueError(
            f"Unsupported review source format: {source_format!r}"
        )

    fields = source_field_maps[source_format]
    raw_review = record.get(fields["review_text"])

    # Extract references from the raw review before cleaning.
    extracted_order_reference = _call_shared_function_or_fallback(
        "extract_order_reference",
        _fallback_extract_order_reference,
        raw_review,
    )
    extracted_product_sku = _call_shared_function_or_fallback(
        "extract_product_sku",
        _fallback_extract_product_sku,
        raw_review,
    )

    # Clean the multilingual review after extraction.
    review_body_clean = _call_shared_function_or_fallback(
        "clean_narrative_text",
        _fallback_clean_review_text,
        raw_review,
    )

    if not isinstance(review_body_clean, str) or review_body_clean == "":
        raise ValueError(
            "clean_narrative_text must return cleaned text or "
            "the literal string 'NaN'."
        )

    if (
        not isinstance(extracted_order_reference, str)
        or extracted_order_reference == ""
    ):
        raise ValueError(
            "extract_order_reference must return a reference or 'NaN'."
        )

    if (
        not isinstance(extracted_product_sku, str)
        or extracted_product_sku == ""
    ):
        raise ValueError(
            "extract_product_sku must return a SKU or 'NaN'."
        )

    # Multilingual fields must be derived from review_body_clean.
    review_body_latin_analysis = _call_shared_function_or_fallback(
        "build_latin_analysis",
        _fallback_build_latin_analysis,
        review_body_clean,
    )
    contains_non_latin_script = _call_shared_function_or_fallback(
        "contains_non_latin_script",
        _fallback_contains_non_latin_script,
        review_body_clean,
    )

    if (
        not isinstance(review_body_latin_analysis, str)
        or review_body_latin_analysis == ""
    ):
        raise ValueError(
            "build_latin_analysis must return text or the literal 'NaN'."
        )

    if type(contains_non_latin_script) is not bool:
        raise ValueError(
            "contains_non_latin_script must return a Python bool."
        )

    if review_body_clean == "NaN":
        review_length_chars = 0
        review_word_count = 0
    else:
        review_length_chars = len(review_body_clean)
        review_word_count = len(review_body_clean.split())

    return {
        "review_id": _normalise_required_review_text(
            record.get(fields["review_id"]),
            fields["review_id"],
        ),
        "order_id": _normalise_required_review_text(
            record.get(fields["order_id"]),
            fields["order_id"],
        ),
        "order_item_id": _normalise_required_review_text(
            record.get(fields["order_item_id"]),
            fields["order_item_id"],
        ),
        "product_id": _normalise_required_review_text(
            record.get(fields["product_id"]),
            fields["product_id"],
        ),
        "customer_id": _normalise_required_review_text(
            record.get(fields["customer_id"]),
            fields["customer_id"],
        ),
        "review_timestamp": _parse_review_timestamp(
            record.get(fields["review_timestamp"]),
            fields["review_timestamp"],
            source_format,
        ),
        "language_code": _normalise_required_review_text(
            record.get(fields["language_code"]),
            fields["language_code"],
        ),
        "rating": _parse_required_review_integer(
            record.get(fields["rating"]),
            fields["rating"],
        ),
        "review_title": _normalise_required_review_text(
            record.get(fields["review_title"]),
            fields["review_title"],
        ),
        "review_body_clean": review_body_clean,
        "review_body_latin_analysis": review_body_latin_analysis,
        "verified_purchase": _parse_review_boolean(
            record.get(fields["verified_purchase"]),
            fields["verified_purchase"],
            source_format,
        ),
        "helpful_votes": _parse_required_review_integer(
            record.get(fields["helpful_votes"]),
            fields["helpful_votes"],
        ),
        "review_length_chars": review_length_chars,
        "review_word_count": review_word_count,
        "contains_non_latin_script": contains_non_latin_script,
        "extracted_order_reference": extracted_order_reference,
        "extracted_product_sku": extracted_product_sku,
        "delivery_experience": _normalise_required_review_text(
            record.get(fields["delivery_experience"]),
            fields["delivery_experience"],
        ),
        "value_experience": _normalise_required_review_text(
            record.get(fields["value_experience"]),
            fields["value_experience"],
        ),
        "writing_style": _normalise_required_review_text(
            record.get(fields["writing_style"]),
            fields["writing_style"],
        ),
    }


# ---------------------------------------------------------------------------
# 5. Reconcile JSON/XML candidates by review_id
# ---------------------------------------------------------------------------

def _reconcile_product_review_candidates(
    candidate_frame,
    output_columns,
):
    """
    Usage:
        canonical, conflicts = _reconcile_product_review_candidates(
            product_review_candidates,
            product_review_columns,
        )

    Reason:
        Within-source duplicates and cross-source overlap must be compared
        field by field after normalisation. Conflicts must be recorded instead
        of applying JSON-over-XML or XML-over-JSON precedence.

    Expected result:
        One canonical row per review_id and a separate conflict DataFrame.
    """
    canonical_rows = []
    conflict_rows = []

    for review_id, group in candidate_frame.groupby(
        "review_id",
        sort=True,
        dropna=False,
    ):
        canonical = {"review_id": review_id}

        for column in output_columns:
            if column == "review_id":
                continue

            non_missing_values = [
                value
                for value in group[column].tolist()
                if not pd.isna(value)
            ]

            distinct_values = list(dict.fromkeys(non_missing_values))

            if len(distinct_values) > 1:
                conflict_rows.append(
                    {
                        "review_id": review_id,
                        "field": column,
                        "normalised_values": distinct_values,
                        "source_rows": (
                            group["_source"]
                            .value_counts()
                            .to_dict()
                        ),
                    }
                )

            canonical[column] = (
                distinct_values[0]
                if distinct_values
                else None
            )

        canonical_rows.append(canonical)

    canonical_frame = pd.DataFrame(
        canonical_rows,
        columns=output_columns,
    )
    conflict_frame = pd.DataFrame(
        conflict_rows,
        columns=[
            "review_id",
            "field",
            "normalised_values",
            "source_rows",
        ],
    )

    return canonical_frame, conflict_frame


# Obtain the exact output field order from the public data dictionary.
product_review_columns = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("product_reviews")
    ]
    .assign(
        _position=lambda frame: frame["position"].astype(int)
    )
    .sort_values("_position")["field_name"]
    .tolist()
)

json_product_review_candidates = pd.DataFrame(
    [
        _standardise_product_review_record(record, "JSON")
        for record in json_reviews
    ],
    columns=product_review_columns,
).assign(_source="JSON")

xml_product_review_candidates = pd.DataFrame(
    [
        _standardise_product_review_record(record, "XML")
        for record in xml_review_records
    ],
    columns=product_review_columns,
).assign(_source="XML")

product_review_candidates = pd.concat(
    [
        json_product_review_candidates,
        xml_product_review_candidates,
    ],
    ignore_index=True,
)

product_reviews, product_review_reconciliation_conflicts = (
    _reconcile_product_review_candidates(
        product_review_candidates,
        product_review_columns,
    )
)

# Stop instead of silently resolving a conflicting normalized value.
if not product_review_reconciliation_conflicts.empty:
    raise ValueError(
        "Conflicting non-missing Product Review values were found:\n"
        + product_review_reconciliation_conflicts.to_string(index=False)
    )

product_reviews = (
    product_reviews.loc[:, product_review_columns]
    .sort_values("review_id", kind="stable")
    .reset_index(drop=True)
)


# ---------------------------------------------------------------------------
# 6. Immediate Task 2 transformation guards
#
# These protect the constructed table but do not replace the later Task 4
# validation register.
# ---------------------------------------------------------------------------

if list(product_reviews.columns) != product_review_columns:
    raise AssertionError(
        "Product Review columns do not match the public data dictionary."
    )

if product_reviews["review_id"].isna().any():
    raise AssertionError(
        "product_reviews.review_id contains missing values."
    )

if not product_reviews["review_id"].is_unique:
    raise AssertionError(
        "product_reviews.review_id is not unique."
    )

if product_reviews.isna().any().any():
    missing_counts = product_reviews.isna().sum()
    missing_counts = missing_counts[missing_counts.gt(0)]

    raise AssertionError(
        "Product Reviews contains prohibited missing values:\n"
        + missing_counts.to_string()
    )

product_review_string_columns = (
    data_dictionary.loc[
        data_dictionary["output_table"].eq("product_reviews")
        & data_dictionary["data_type"].eq("string"),
        "field_name",
    ]
    .tolist()
)

blank_string_counts = {
    column: int(
        product_reviews[column]
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )
    for column in product_review_string_columns
}
blank_string_counts = {
    column: count
    for column, count in blank_string_counts.items()
    if count > 0
}

if blank_string_counts:
    raise AssertionError(
        "Product Reviews contains empty required strings: "
        f"{blank_string_counts}"
    )

for boolean_column in [
    "verified_purchase",
    "contains_non_latin_script",
]:
    invalid_boolean_count = int(
        product_reviews[boolean_column]
        .map(lambda value: type(value) is not bool)
        .sum()
    )

    if invalid_boolean_count:
        raise AssertionError(
            f"{boolean_column} contains "
            f"{invalid_boolean_count} non-boolean values."
        )

invalid_timestamp_count = int(
    (
        ~product_reviews["review_timestamp"].str.fullmatch(
            r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"
        )
    ).sum()
)

if invalid_timestamp_count:
    raise AssertionError(
        f"Product Reviews contains {invalid_timestamp_count} "
        "invalid timestamp formats."
    )


# Check the Product foreign key now because your products table is complete.
# The remaining parent-table checks require teammates' standardised tables.
if "products" in globals():
    missing_product_fk_values = sorted(
        set(product_reviews["product_id"])
        - set(products["product_id"])
    )

    if missing_product_fk_values:
        raise AssertionError(
            "Product Review product_id values are missing from products: "
            f"{missing_product_fk_values[:10]}"
        )

    product_fk_status = "PASS"
else:
    missing_product_fk_values = []
    product_fk_status = "PENDING - run the Product cell first"


# ---------------------------------------------------------------------------
# 7. Task 2 row-flow and preview evidence
# ---------------------------------------------------------------------------

product_review_row_flow = pd.DataFrame(
    [
        {
            "stage": "structured JSON review rows",
            "rows": len(json_reviews),
        },
        {
            "stage": "structured XML review rows",
            "rows": len(xml_review_records),
        },
        {
            "stage": "combined normalised candidates",
            "rows": len(product_review_candidates),
        },
        {
            "stage": "canonical product reviews",
            "rows": len(product_reviews),
        },
    ]
)

show(
    product_review_row_flow,
    "Product Review transformation row flow",
)

product_review_preview = product_reviews[
    [
        "review_id",
        "order_id",
        "order_item_id",
        "product_id",
        "customer_id",
        "review_timestamp",
        "language_code",
        "rating",
        "verified_purchase",
        "helpful_votes",
        "review_length_chars",
        "review_word_count",
        "contains_non_latin_script",
        "extracted_order_reference",
        "extracted_product_sku",
    ]
].head()

show(
    product_review_preview,
    "Standardised Product Review preview",
)

product_review_function_sources = pd.DataFrame(
    [
        {
            "required function": function_name,
            "current source": (
                "shared teammate function"
                if callable(globals().get(function_name))
                else "private Task 2 fallback"
            ),
        }
        for function_name in [
            "clean_narrative_text",
            "extract_order_reference",
            "extract_product_sku",
            "build_latin_analysis",
            "contains_non_latin_script",
        ]
    ]
)

show(
    product_review_function_sources,
    "Product Review function integration status",
)

print(
    "\nProduct Review table ready:"
    f" rows={len(product_reviews)},"
    f" columns={len(product_reviews.columns)},"
    f" reconciliation_conflicts="
    f"{len(product_review_reconciliation_conflicts)},"
    f" product_fk={product_fk_status}"
)


Product Review transformation row flow
                         stage  rows
   structured JSON review rows  3946
    structured XML review rows  3946
combined normalised candidates  7892
     canonical product reviews  7000

Standardised Product Review preview
 review_id   order_id order_item_id product_id customer_id    review_timestamp language_code  rating  verified_purchase  helpful_votes  review_length_chars  review_word_count  contains_non_latin_script extracted_order_reference extracted_product_sku
HREV000001 HORD000001   HITM0000001    PRD0837    CUS00191 2018-12-16 11:01:00            ru       4               True              4                 1398                210                       True                HORD000001          SKU-CAN00837
HREV000002 HORD000002   HITM0000004    PRD0726    CUS00096 2018-05-23 12:02:00            en       5               True             15                  741                120                      False                HORD000002          S

## 5. Reconcile overlap and verify relationships

Demonstrate how records are compared by stable business key, how canonical
rows are retained and how silent source-precedence choices are avoided.


## 6. Validation register

Keep each check executable and give it a stable `VAL-...` ID. Immediately after
each code check, record the observed result, `PASS`/`FAIL`, evidence and
resolution/interpretation. A genuine, explained failure is preferable to a
fabricated pass.

Required areas include schema/types, primary and foreign keys, row flow and
source coverage, overlap, arithmetic, temporal logic, text/reference behaviour
and multilingual handling.


### 6.1 Schema and type checks (`VAL-SCHEMA-...`)


**Observed result/status/interpretation:** Replace.


### 6.2 Primary- and foreign-key checks (`VAL-PK-...`, `VAL-FK-...`)


**Observed result/status/interpretation:** Replace.


### 6.3 Source coverage and reconciliation checks (`VAL-FLOW-...`)


**Observed result/status/interpretation:** Replace.


### 6.4 Arithmetic checks (`VAL-ARITH-...`)


**Observed result/status/interpretation:** Replace.


### 6.5 Temporal checks (`VAL-TIME-...`)


**Observed result/status/interpretation:** Replace.


### 6.6 Text and multilingual checks (`VAL-TEXT-...`)


**Observed result/status/interpretation:** Replace.


### 6.7 Literal `NaN` reminder

For prescribed missing string outputs, the expected value is the three text
characters `NaN`, not an empty field, Python `None` or a floating-point NaN.
Use `pandas.read_csv(path, keep_default_na=False)` when validating that sentinel.


## 7. Export the six CSV files

Export exactly the required filenames, columns and order. Display a compact
final schema/row-count summary without hard-coding certified counts.


## 8. Final reproducibility record

Record the final run date, dependency versions and the result of Restart and Run
All. Confirm that the six outputs and validation evidence were recreated from
the allocated raw files.
